# ASG Airlines — Data Profiling

## Objective
Examine the source dataset before applying any cleaning or transformations.

This notebook identifies:
- Available tables and columns
- Record counts and data types
- Missing values
- Exact duplicate records
- Repeated identifiers

The original workbook is preserved unchanged.

The reusable implementation now lives in `src/`. Each section calls a named shared function, then retains the profiling summaries and teaching checks below. The CLI follows the same sequence; there is no second set of cleaning rules. See [README](../README.md) for setup, the single-command pipeline, synthetic tests, and snapshot regression checks.


In [1]:
from pathlib import Path
import sys
import pandas as pd

# Run Jupyter from the checkout root (or its notebooks directory).
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "pipeline.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src" / "pipeline.py").is_file(), "Open this notebook from the project"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import cleaning, privacy, model, persistence
from src.ingestion import load_workbook, validate_source_schema
from src.pipeline import new_run_summary, quality_summary

SOURCE_FILE = PROJECT_ROOT / "data/raw/UseCase - Airlines.xlsx"
KEY_FILE = PROJECT_ROOT / ".secrets/passenger_hmac.key"
notebook_run = new_run_summary(SOURCE_FILE)
raw_tables = load_workbook(SOURCE_FILE)
notebook_run["input"]["sha256"] = persistence.sha256_file(SOURCE_FILE)
print("Workbook loaded successfully.")
print("Tables:", list(raw_tables))


Workbook loaded successfully.
Tables: ['flights', 'payments', 'bookings', 'passengers']


In [2]:
raw_tables["flights"]

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00
...,...,...,...,...,...,...,...
1015,6F198,IndiGo,HYD,BLR,2026-04-17 12:40:41.703,2026-04-17 17:39:41.703,04:59:00
1016,AI093,Air India,BOM,BLR,2026-04-17 12:35:41.702,2026-04-17 17:28:41.702,04:53:00
1017,UK062,Vistara,HYD,DEL,2026-04-17 12:33:41.701,2026-04-17 15:35:41.701,03:02:00
1018,6F057,IndiGo,MAA,DEL,2026-04-17 12:32:41.701,2026-04-17 15:50:41.701,03:18:00


In [3]:
validate_source_schema(raw_tables)
print("Required source sheets and columns are present.")


Required source sheets and columns are present.


In [4]:
primary_keys = {
    "flights": "flight_id",
    "bookings": "booking_id",
    "payments": "payment_id",
    "passengers": "passenger_id",
}

profile_rows = []

for table_name, key_column in primary_keys.items():
    df = raw_tables[table_name]

    if key_column not in df.columns:
        raise ValueError(
            f"{table_name}: required key column "
            f"'{key_column}' is missing."
        )

    non_missing_keys = df[key_column].dropna()

    profile_rows.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "exact_duplicate_excess": int(df.duplicated().sum()),
        "missing_keys": int(df[key_column].isna().sum()),
        "distinct_keys": int(non_missing_keys.nunique()),
        "repeated_key_excess": int(
            non_missing_keys.duplicated().sum()
        ),
    })

profile_summary = pd.DataFrame(profile_rows)

display(profile_summary)

,table,rows,columns,missing_cells,exact_duplicate_excess,missing_keys,distinct_keys,repeated_key_excess
0,flights,1020,7,41,15,0,1004,16
1,bookings,1000,9,45,0,0,1000,0
2,payments,1000,4,48,0,0,1000,0
3,passengers,1039,9,10,0,0,1000,39


In [5]:
column_profile_rows = []

for table_name, df in raw_tables.items():
    for column in df.columns:
        missing_count = int(df[column].isna().sum())

        column_profile_rows.append({
            "table": table_name,
            "column": column,
            "data_type": str(df[column].dtype),
            "missing_count": missing_count,
            "missing_percent": round(
                missing_count / len(df) * 100, 2
            ) if len(df) else 0.0,
        })

column_profile = pd.DataFrame(column_profile_rows)

display(column_profile)

,table,column,data_type,missing_count,missing_percent
0,flights,flight_id,string,0,0.00
1,flights,airline,str,41,4.02
2,flights,source,str,0,0.00
3,flights,destination,str,0,0.00
4,flights,departure_time,datetime64[us],0,0.00
5,flights,arrival_time,datetime64[us],0,0.00
6,flights,duration,object,0,0.00
7,payments,payment_id,string,0,0.00
8,payments,booking_id,string,0,0.00
9,payments,amount,object,48,4.80


In [6]:
display(
    column_profile.loc[
        column_profile["missing_count"] > 0,
        ["table", "column", "missing_count", "missing_percent"],
    ].reset_index(drop=True)
)

,table,column,missing_count,missing_percent
0,flights,airline,41,4.02
1,payments,amount,48,4.80
2,bookings,status,45,4.50
3,passengers,last_name,10,0.96


In [7]:
# The shared function preserves raw row indexes as lineage.
flights_raw = raw_tables["flights"].copy()
flights_removed_duplicates, flights_unique_rows = cleaning.split_exact_duplicates(flights_raw)
exact_duplicate_mask = flights_raw.index.isin(flights_removed_duplicates.index)
print("Original flight records:", len(flights_raw))
print("Excess exact duplicates:", len(flights_removed_duplicates))


Original flight records: 1020
Excess exact duplicates: 15


In [8]:
print("Flights after removing excess exact copies:", len(flights_unique_rows))


Flights after removing excess exact copies: 1005


In [9]:
flight_checks = cleaning.check_flights(flights_unique_rows)
conflicting_id_mask = flight_checks["flight_id_conflict"]
flight_id_conflicts = flights_unique_rows.loc[conflicting_id_mask].sort_values(
    ["flight_id", "departure_time"]
)
print("Repeated flight IDs:", flight_id_conflicts["flight_id"].nunique())
print("Records sharing those IDs:", len(flight_id_conflicts))


Repeated flight IDs: 1
Records sharing those IDs: 2


## Flight timestamp validation

Work on `flights_unique_rows`, after removing excess exact duplicate copies. Keep the original `departure_time`, `arrival_time`, and source `duration` columns unchanged; add `departure_ts`, `arrival_ts`, and a calculated `duration_minutes` to a separate working copy.

**Timezone assumption:** departure and arrival timestamps share a common timezone. The source timestamps do not identify timezones, so we do not infer airport offsets or convert them. Elapsed times are provisional until this assumption is confirmed.

`pd.to_datetime(..., errors="coerce")` converts missing or unparseable values to `NaT` (not a time). Subtract the full timestamps and use `.dt.total_seconds() / 60`: this preserves the sign and all days. `.dt.seconds` would discard the day component and can hide negative durations. Never add 24 hours to a negative result: the source already includes full dates.

- `timestamp_parse_issue`: at least one timestamp is missing or unparseable; the elapsed time cannot be established.
- `non_positive_duration`: the calculated duration is zero or negative. A missing duration is handled by the timestamp flag.
- `flight_id_conflict`: multiple different records share a flight ID after exact duplicate removal. `keep=False` flags every member; there is no evidence to choose a winner.
- `is_overnight`: arrival is on a later calendar date, using `.dt.normalize()` to compare dates at midnight. This includes arrivals more than one day later and is informational, not a quarantine reason.


In [10]:
# Parsed timestamps live beside the original values; the helper never repairs dates.
print("Timestamp parsing issues:", int(flight_checks["timestamp_parse_issue"].sum()))


Timestamp parsing issues: 0


In [11]:
# Signed total_seconds() / 60 and calendar-date comparison are implemented in check_flights.
display(flight_checks[["timestamp_parse_issue", "non_positive_duration", "is_overnight"]]
        .sum().astype(int).to_frame("flight_records"))


,flight_records
timestamp_parse_issue,0
non_positive_duration,1
is_overnight,122


In [12]:
time_issue_mask = (
    flight_checks["timestamp_parse_issue"]
    | flight_checks["non_positive_duration"]
)

display(
    flight_checks.loc[
        time_issue_mask,
        [
            "flight_id",
            "source",
            "destination",
            "departure_time",
            "arrival_time",
            "duration",
            "duration_minutes",
            "timestamp_parse_issue",
            "non_positive_duration",
        ],
    ]
)

,flight_id,source,destination,departure_time,arrival_time,duration,duration_minutes,timestamp_parse_issue,non_positive_duration
355,SJ192,HYD,BOM,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00,-1140.0,False,True


## Flight record classification and reconciliation

Create three mutually exclusive in-memory outputs, with no workbook edits or exports:

1. `flights_removed_duplicates`: only excess identical copies, keeping the first occurrence for validation. If that retained occurrence has an issue, it will be quarantined below.
2. `flights_quarantined`: remaining records with any timestamp or ambiguous-ID issue. `issue_reasons` is a list so overlapping reasons are all retained. Quarantine means hold for clarification, not delete or guess a correction.
3. `flights_accepted`: remaining records without those issues, accepted **provisionally** for this stage only. Overnight arrival alone is valid.

Keep the original DataFrame index to trace each output back to a distinct raw row. Reconciliation checks both counts and original row contents, because a matching total alone could conceal a missing row and a repeated row. Flag counts may overlap and must not be added to count quarantined records. Duration statistics use only `flights_accepted`; the source `duration` field is not used to override timestamp arithmetic.


In [13]:
issue_labels = cleaning.issue_labels
flights_quarantined, flights_accepted = cleaning.partition_flights(flight_checks)
quarantine_mask = flight_checks.index.isin(flights_quarantined.index)
issue_flags = flight_checks[list(issue_labels)]
display(issue_flags.sum().astype(int).to_frame("flagged_records"))


,flagged_records
timestamp_parse_issue,0
non_positive_duration,1
flight_id_conflict,2


In [14]:
flight_reconciliation = pd.Series({
    "Raw rows": len(flights_raw),
    "Removed exact duplicate copies": len(flights_removed_duplicates),
    "Quarantined records": len(flights_quarantined),
    "Provisionally accepted records": len(flights_accepted),
}, name="rows")
output_total = flight_reconciliation.iloc[1:].sum()
display(flight_reconciliation.to_frame())
print("Raw rows minus output rows:", len(flights_raw) - output_total)

print("Accepted overnight flights:", int(flights_accepted["is_overnight"].sum()))
accepted_duration_summary = flights_accepted["duration_minutes"].agg(
    ["count", "mean", "median", "min", "max"]
)
display(accepted_duration_summary.to_frame(name="accepted_duration_minutes"))

# Check row identity and preserved raw values, not expected checkpoint numbers.
reconciled_flights = pd.concat([
    flights_removed_duplicates[flights_raw.columns],
    flights_quarantined[flights_raw.columns],
    flights_accepted[flights_raw.columns],
]).sort_index()
assert reconciled_flights.index.is_unique, "A raw row appears in multiple outputs"
assert output_total == len(flights_raw), "Row counts do not reconcile"
pd.testing.assert_frame_equal(reconciled_flights, flights_raw.sort_index())
pd.testing.assert_frame_equal(flights_raw, raw_tables["flights"])
assert flights_accepted["duration_minutes"].gt(0).all()
assert not flights_accepted["flight_id"].duplicated().any()
assert not flights_accepted[list(issue_labels)].any(axis=None)
assert flights_quarantined["issue_reasons"].map(bool).all()
# Every flagged issue must survive in the list, including overlapping issues.
for flag, reason in issue_labels.items():
    assert flight_checks["issue_reasons"].map(
        lambda reasons: reason in reasons
    ).equals(flight_checks[flag])

import sys
assert sys.prefix == str(PROJECT_ROOT / ".venv"), "Use the project environment"
print("Checks passed. Python interpreter:", sys.executable)


,rows
Raw rows,1020
Removed exact duplicate copies,15
Quarantined records,3
Provisionally accepted records,1002


Raw rows minus output rows: 0
Accepted overnight flights: 122


,accepted_duration_minutes
count,1002.00000
mean,164.53706
median,166.00000
min,30.00000
max,300.00000


Checks passed. Python interpreter: /home/rrm/NeoStats/asg-airlines/.venv/bin/python


### Timestamp stage completed

The outputs above are computed from the workbook, not adjusted to match expected checkpoints. A negative duration needs source clarification; ambiguous IDs need an authoritative way to distinguish the flights before joining other tables. Passing these checks does not validate routes, other flight attributes, or relationships to bookings and payments.

The next section standardizes airline and route fields in the provisionally accepted flights. Cleaning other tables and Power BI work remain for later stages. No passenger personal information is displayed. These tables exist in memory when the notebook runs; the saved notebook contains code and results, and rerunning it recreates the tables.


## Standardize accepted flight airline and route fields

Start with `flights_accepted.copy()` and retain its original row index for lineage. Preserve `flight_id`, `source`, `destination`, and `airline` in matching `_raw` columns before changing the reporting fields. All existing timestamps, durations, quality flags, and issue reasons remain unchanged.

Use pandas nullable `string` dtype, `.str.strip()`, and `.str.upper()` for IDs and airport codes. Unlike `astype(str)`, nullable strings preserve missing values instead of creating literal text such as `nan`. Treat whitespace-only IDs and endpoints as missing too.

For airline names, trim whitespace and use `.str.casefold()` for case-insensitive matching to Air India, IndiGo, SpiceJet, and Vistara. The `airline` column is the reporting label; `airline_raw` keeps the source value. Missing, blank, and explicit UNKNOWN labels report as `UNKNOWN`, with separate flags:

- `airline_missing`: the loaded source value is null.
- `airline_blank`: a non-null source value contains only whitespace or is empty.
- `airline_explicit_unknown`: the trimmed source label says UNKNOWN, ignoring case.
- `airline_unrecognized`: any other label outside the four canonical names. Keep its trimmed text visible and flag it for review.

These flags describe values as loaded by the existing Excel reader; Excel empty cells already loaded as null cannot subsequently be distinguished from other nulls. Unknown-airline flights remain usable for flight counts, routes, and duration analysis because a missing airline name does not invalidate those fields. Include UNKNOWN in airline summaries so totals reconcile. Never guess names from flight ID prefixes: this dataset has not established that relationship.

Validation flags missing IDs/endpoints, non-missing IDs outside `(AI|SJ|UK|6F)[0-9]{3}`, repeated normalized IDs, identical source/destination, and non-missing codes outside BLR, BOM, CCU, DEL, HYD, MAA. The ID pattern is dataset-specific, not a universal airline identifier rule; the working airport allowlist is not a complete airport registry. Accepted IDs were unique before normalization, so any repeated normalized ID is a newly introduced conflict; flag every member. New flags supplement the earlier quality flags without changing the earlier quarantine decision or removing rows.

Build directional `route` only when both endpoints exist: DEL → BOM and BOM → DEL are different routes. `departure_date` and `departure_hour` come from the previously parsed `departure_ts`, under the existing common-timezone assumption. `duration_hours` is `duration_minutes / 60`, with no rounding of stored values.


In [15]:
flights_standardized = cleaning.standardize_flights(flights_accepted)
standardized_fields = cleaning.standardized_fields
canonical_airlines = cleaning.canonical_airlines
unknown_airline_mask = flights_standardized[
    ["airline_missing", "airline_blank", "airline_explicit_unknown"]
].any(axis=1)


In [16]:
# These diagnostics are added by standardize_flights; review flags, do not drop rows.
airport_allowlist = cleaning.airport_allowlist
print("Standardized flight fields and dataset-specific route/ID checks created.")


Standardized flight fields and dataset-specific route/ID checks created.


In [17]:
# Explicitly include UNKNOWN even if a future run has zero unknown labels.
airline_counts = flights_standardized["airline"].value_counts()
airline_counts = airline_counts.reindex(
    pd.Index([*canonical_airlines.values(), "UNKNOWN"]).union(airline_counts.index, sort=False),
    fill_value=0,
).rename_axis("airline").rename("flight_count")
display(airline_counts.to_frame())

standardization_flags = [
    "airline_missing", "airline_blank", "airline_explicit_unknown", "airline_unrecognized",
    "flight_id_missing", "source_missing", "destination_missing",
    "flight_id_format_issue", "flight_id_normalization_conflict",
    "source_equals_destination", "source_outside_allowlist", "destination_outside_allowlist",
]
standardization_flag_counts = flights_standardized[standardization_flags].sum().astype(int)
display(standardization_flag_counts.to_frame(name="flagged_records"))

directional_route_counts = flights_standardized["route"].value_counts().rename("flight_count")
print("Distinct directional routes:", flights_standardized["route"].nunique())
# Alphabetical order makes equal-count ties deterministic.
top_10_routes = directional_route_counts.sort_index().sort_values(
    ascending=False, kind="stable"
).head(10)
display(top_10_routes.to_frame())

# Unknown airline categories are summarized above; show other issues if present.
standardization_review_flags = standardization_flags[3:]
standardization_review_mask = flights_standardized[standardization_review_flags].any(axis=1)
print("Records needing additional field review:", int(standardization_review_mask.sum()))
if standardization_review_mask.any():
    display(flights_standardized.loc[standardization_review_mask, [
        "flight_id_raw", "source_raw", "destination_raw", "airline_raw",
        "flight_id", "source", "destination", "airline", *standardization_review_flags,
    ]])


,flight_count
airline,
Air India,233
IndiGo,249
SpiceJet,235
Vistara,218
UNKNOWN,67


,flagged_records
airline_missing,39
airline_blank,0
airline_explicit_unknown,28
airline_unrecognized,0
flight_id_missing,0
source_missing,0
destination_missing,0
flight_id_format_issue,0
flight_id_normalization_conflict,0
source_equals_destination,0


Distinct directional routes: 30


,flight_count
route,
BOM → CCU,90
CCU → DEL,72
MAA → BLR,65
BLR → BOM,60
HYD → MAA,57
DEL → HYD,54
HYD → DEL,42
BOM → DEL,39
CCU → BOM,33


Records needing additional field review: 0


In [18]:
# Preserve every accepted row and every existing field except the four reporting fields.
pd.testing.assert_index_equal(flights_standardized.index, flights_accepted.index)
unchanged_columns = flights_accepted.columns.difference(standardized_fields)
pd.testing.assert_frame_equal(
    flights_standardized[unchanged_columns], flights_accepted[unchanged_columns]
)
for column in standardized_fields:
    pd.testing.assert_series_equal(
        flights_standardized[f"{column}_raw"], flights_accepted[column], check_names=False
    )
pd.testing.assert_frame_equal(flights_accepted, flight_checks.loc[~quarantine_mask])
pd.testing.assert_frame_equal(flights_raw, raw_tables["flights"])

# Missing values must survive normalization, and derived totals must reconcile.
for column in ["flight_id", "source", "destination"]:
    expected_missing = (
        flights_accepted[column].isna()
        | flights_accepted[column].astype("string").str.strip().eq("").fillna(False)
    )
    assert flights_standardized[column].isna().eq(expected_missing).all()
assert flights_standardized["airline"].eq("UNKNOWN").eq(unknown_airline_mask).all()
assert airline_counts.sum() == len(flights_accepted)
assert flights_standardized["route"].notna().equals(
    flights_standardized[["source", "destination"]].notna().all(axis=1)
)
assert directional_route_counts.sum() == flights_standardized["route"].notna().sum()
pd.testing.assert_series_equal(
    flights_standardized["duration_hours"],
    flights_accepted["duration_minutes"] / 60, check_names=False,
)
print("Preservation checks passed. Standardized records:", len(flights_standardized))
print("Overnight flights unchanged:", int(flights_standardized["is_overnight"].sum()))
print("Mean duration in minutes:", flights_standardized["duration_minutes"].mean())
print("Missing source airline values:", int(flights_standardized["airline_missing"].sum()))
print("Blank source airline values:", int(flights_standardized["airline_blank"].sum()))
print("Explicit UNKNOWN source values:", int(flights_standardized["airline_explicit_unknown"].sum()))
print("UNKNOWN reporting airline values:", int(flights_standardized["airline"].eq("UNKNOWN").sum()))


Preservation checks passed. Standardized records: 1002
Overnight flights unchanged: 122
Mean duration in minutes: 164.53705998003994
Missing source airline values: 39
Blank source airline values: 0
Explicit UNKNOWN source values: 28
UNKNOWN reporting airline values: 67


### Flight standardization completed

All summaries above are calculated from the accepted flights, not hardcoded checkpoints. No new flags silently exclude records, and the earlier removed-duplicate and quarantine tables are unchanged. `flights_standardized` remains provisional; UNKNOWN is a reporting category, not a recovered airline name. Flag counts can overlap, so they are not additive counts of affected flights.

The following stage standardizes bookings and validates their references. Payments and passenger records remain unchanged, no final datasets are exported, and no dashboard is built. Rerun the notebook to recreate the in-memory standardized tables.


## Booking standardization and relationship validation

The intended grain is **one row per booking**. Start from a copy of `raw_tables["bookings"]`; do not remove exact copies or choose among repeated booking IDs. Preserve changed fields in `*_raw` columns, keep the original index, and add `booking_source_row` (Excel row number: the loaded zero-based index + 2, accounting for the header). Other source fields remain untouched. Booking rows contain personal information: display aggregate summaries only, with explicitly selected safe columns for any diagnostic grouping.

Normalize booking, passenger, and flight IDs with nullable pandas strings, whitespace trimming, and uppercase. Separate `*_missing` (null as loaded) from `*_blank` (non-null but empty after trimming); both become missing reporting identifiers. A normalization collision means **different original identifier strings collapse to one non-missing key**. Repeated passenger or flight IDs across bookings are expected references, not duplicate bookings. Exact-duplicate flags compare all original booking columns before adding lineage or derived fields; distinguish all duplicate-group members from excess copies.

Recognized reporting statuses are CONFIRMED, CANCELLED, and PENDING. Preserve `status_raw`; map missing, blank, and all unrecognized labels to UNKNOWN, with separate flags. UNKNOWN remains in the denominator and visible in summaries rather than silently making known statuses appear more complete. No status is inferred from payments. Missing/blank distinctions describe what the existing Excel reader loaded; original blank cells may already have become null.

Keep `booking_date` unchanged and parse `booking_ts` separately with `errors="coerce"`. Report missing, blank, unparseable, and combined timestamp issues. Comparing booking and departure timestamps assumes that they share a common timezone, as in the flight stage.


In [19]:
normalize_identifier = cleaning.normalize_identifier
identifier_normalization_collisions = cleaning.identifier_normalization_collisions
bookings_raw = raw_tables["bookings"].copy()
booking_identifier_columns = ["booking_id", "passenger_id", "flight_id"]
recognized_booking_statuses = cleaning.recognized_booking_statuses
bookings_standardized, booking_identifier_profile = cleaning.standardize_bookings(bookings_raw)


### Flight references and a merge that preserves coverage

Build the source lookup from `flight_checks`, which already excludes excess exact flight copies and retains the earlier quality flags. Count source records by normalized flight ID, and separately count distinct original spellings to detect normalization collisions. An ID can exist yet identify multiple different source flights; existence alone does not establish a unique match.

Assign each booking exactly one `flight_reference_status`:

- `missing`: no normalized flight ID.
- `ambiguous`: more than one deduplicated source flight has that normalized ID, including collisions introduced by normalization.
- `accepted`: exactly one source flight, present in `flights_standardized`, with no earlier quarantine flag.
- `quarantined`: one source flight excluded by the earlier quality rules.
- `unmatched`: the identifier is absent from the source flight lookup.

Only accepted references enter the flight-attribute lookup. A left merge keeps every booking, including unresolved references; `validate="many_to_one"` rejects a lookup that could multiply rows. Retain lineage and verify row count and order. Missing keys are excluded from the lookup so pandas cannot match a missing booking key to a missing flight key. Unresolved references receive no flight attributes. They cannot be attributed to routes yet, but remain in overall booking and status counts.

For accepted matches with valid timestamps, `booking_after_departure` is True when booking occurs strictly after departure and False when at or before departure. All unevaluable rows retain nullable boolean `pd.NA`, meaning unknown rather than passed. Report anomalies without guessing corrected dates.


In [20]:
(
    bookings_standardized, flight_reference_lookup, source_flight_references,
    accepted_flight_lookup, booking_date_evaluable,
) = cleaning.attach_flight_references(bookings_standardized, flight_checks, flights_standardized)
flight_attribute_columns = ["departure_ts", "airline", "source", "destination", "route"]


### Passenger existence without joining personal data

Read only the passenger ID column from the already loaded passenger table. Normalize a separate identifier Series for reference checking; do not modify or deduplicate passengers. Count all source occurrences per normalized ID.

Classify references as `missing`, `unmatched`, `existing-unique` (one source occurrence), or `existing-nonunique` (multiple source occurrences). The last category means identity resolution is pending, not that the booking is automatically invalid. Even identical passenger copies still count as nonunique at this stage. Report normalization-collision counts separately. No passenger rows are merged onto bookings and no passenger identifiers or personal fields are displayed.


In [21]:
bookings_standardized = cleaning.attach_passenger_references(
    bookings_standardized, raw_tables["passengers"]["passenger_id"]
)
# Identifier-only diagnostics; never join or display passenger source records.
passenger_reference_ids_raw = raw_tables["passengers"]["passenger_id"].copy()
passenger_reference_ids = normalize_identifier(passenger_reference_ids_raw)
passenger_reference_counts = passenger_reference_ids.value_counts()
passenger_source_collisions = identifier_normalization_collisions(passenger_reference_ids_raw)


### Booking summaries and observed shares

All booking counts below use source rows, retaining duplicates and unresolved references if present. Duplicate flags count all group members unless explicitly labeled excess; identifier collision flags count every member whose key has multiple original spellings. Flags can overlap and should not be added to obtain a distinct affected-row count.

The **observed cancelled-booking share** is CANCELLED rows divided by all booking rows. Show the UNKNOWN-status share alongside it: this is an observed row share, not a cancellation rate among only known statuses or an inferred rate for unresolved statuses. No payment information is used.


In [22]:
booking_row_summary = pd.Series({
    "Raw booking rows": len(bookings_raw),
    "Standardized booking rows after flight merge": len(bookings_standardized),
    "Booking rows added by merge": len(bookings_standardized) - len(bookings_raw),
}, name="rows")
display(booking_row_summary.to_frame())
display(booking_identifier_profile)

booking_duplicate_flags = [
    "booking_exact_duplicate", "booking_exact_duplicate_excess",
    "booking_id_repeated_raw", "booking_id_repeated",
]
display(bookings_standardized[booking_duplicate_flags].sum().astype(int).to_frame("flagged_rows"))

booking_status_counts = bookings_standardized["status"].value_counts().reindex(
    [*recognized_booking_statuses, "UNKNOWN"], fill_value=0
)
display(booking_status_counts.rename_axis("status").to_frame("booking_rows"))
booking_status_flags = ["status_missing", "status_blank", "status_unrecognized"]
display(bookings_standardized[booking_status_flags].sum().astype(int).to_frame("flagged_rows"))

flight_reference_counts = bookings_standardized["flight_reference_status"].value_counts().reindex(
    ["missing", "accepted", "ambiguous", "quarantined", "unmatched"], fill_value=0
)
passenger_booking_reference_counts = bookings_standardized["passenger_reference_status"].value_counts().reindex(
    ["missing", "unmatched", "existing-unique", "existing-nonunique"], fill_value=0
)
display(flight_reference_counts.rename_axis("flight_reference_status").to_frame("booking_rows"))
display(passenger_booking_reference_counts.rename_axis("passenger_reference_status").to_frame("booking_rows"))

reference_lookup_summary = pd.Series({
    "Deduplicated source flight rows": len(source_flight_references),
    "Source flight rows without a normalized ID": int(source_flight_references["flight_id"].isna().sum()),
    "Ambiguous normalized source flight IDs": int(flight_reference_lookup["source_records"].gt(1).sum()),
    "Source flight normalization collision keys": int(flight_reference_lookup["normalization_collision"].sum()),
    "Source flight normalization collision rows": int(flight_reference_lookup.loc[
        flight_reference_lookup["normalization_collision"], "source_records"
    ].sum()),
    "Passenger source ID rows": len(passenger_reference_ids),
    "Passenger source rows without a normalized ID": int(passenger_reference_ids.isna().sum()),
    "Nonunique normalized passenger keys": int(passenger_reference_counts.gt(1).sum()),
    "Passenger source normalization collision rows": int(passenger_source_collisions.sum()),
    "Passenger source normalization collision keys": passenger_reference_ids[passenger_source_collisions].nunique(),
}, name="count")
display(reference_lookup_summary.to_frame())

# Only safe flight identifiers and classifications are used in this diagnostic aggregate.
unresolved_flight_reference_counts = (
    bookings_standardized.loc[
        ~bookings_standardized["flight_reference_status"].eq("accepted"),
        ["flight_id", "flight_reference_status"],
    ].value_counts(dropna=False).rename("booking_rows")
)
if not unresolved_flight_reference_counts.empty:
    display(unresolved_flight_reference_counts.to_frame())

booking_date_flags = [
    "booking_date_missing", "booking_date_blank",
    "booking_date_unparseable", "booking_date_parse_issue",
]
booking_date_issue_counts = bookings_standardized[booking_date_flags].sum().astype(int)
booking_date_issue_counts["booking_after_departure"] = int(bookings_standardized["booking_after_departure"].sum())
booking_date_issue_counts["booking_at_or_before_departure"] = int(bookings_standardized["booking_after_departure"].eq(False).sum())
booking_date_issue_counts["booking_departure_check_unknown"] = int(bookings_standardized["booking_after_departure"].isna().sum())
display(booking_date_issue_counts.to_frame("booking_rows"))

booking_observed_shares = pd.Series({
    "Observed cancelled-booking share": bookings_standardized["status"].eq("CANCELLED").mean(),
    "Unknown-status share": bookings_standardized["status"].eq("UNKNOWN").mean(),
}, name="share")
display((booking_observed_shares * 100).to_frame("percent_of_all_booking_rows"))


,rows
Raw booking rows,1000
Standardized booking rows after flight merge,1000
Booking rows added by merge,0


,raw_distinct_nonmissing,standardized_distinct_nonmissing,missing,blank,normalization_collision_rows,normalization_collision_keys
identifier,,,,,,
booking_id,1000,1000,0,0,0,0
passenger_id,636,636,0,0,0,0
flight_id,984,984,0,0,0,0


,flagged_rows
booking_exact_duplicate,0
booking_exact_duplicate_excess,0
booking_id_repeated_raw,0
booking_id_repeated,0


,booking_rows
status,
CONFIRMED,320
CANCELLED,314
PENDING,291
UNKNOWN,75


,flagged_rows
status_missing,45
status_blank,0
status_unrecognized,30


...

,booking_rows
passenger_reference_status,
missing,0
unmatched,0
existing-unique,966
existing-nonunique,34


,count
Deduplicated source flight rows,1005
Source flight rows without a normalized ID,0
Ambiguous normalized source flight IDs,1
Source flight normalization collision keys,0
Source flight normalization collision rows,0
Passenger source ID rows,1039
Passenger source rows without a normalized ID,0
Nonunique normalized passenger keys,36
Passenger source normalization collision rows,0
Passenger source normalization collision keys,0


,,booking_rows
flight_id,flight_reference_status,
6F250,ambiguous,2
SJ192,quarantined,1


,booking_rows
booking_date_missing,0
booking_date_blank,0
booking_date_unparseable,0
booking_date_parse_issue,0
booking_after_departure,0
booking_at_or_before_departure,997
booking_departure_check_unknown,3


,percent_of_all_booking_rows
Observed cancelled-booking share,31.4
Unknown-status share,7.5


In [23]:
# Small edge check for the identifier operations reused by all three tables.
identifier_example = pd.Series([None, " ", " ai001 ", "AI001", "p002"], dtype="string")
assert normalize_identifier(identifier_example).fillna("<missing>").tolist() == [
    "<missing>", "<missing>", "AI001", "AI001", "P002",
]
assert identifier_normalization_collisions(identifier_example).tolist() == [
    False, False, True, True, False,
]

# Reconstruct every original booking column without printing personal data on failure.
booking_original_columns = bookings_standardized[bookings_raw.columns].copy()
for column in [*booking_identifier_columns, "status"]:
    booking_original_columns[column] = bookings_standardized[f"{column}_raw"]
assert booking_original_columns.equals(bookings_raw), "Original booking values or lineage changed"
assert bookings_standardized["booking_source_row"].tolist() == (bookings_raw.index + 2).tolist()
assert bookings_standardized["booking_source_row"].is_unique
assert len(bookings_standardized) == len(bookings_raw)
assert len(source_flight_references) == len(flights_unique_rows)
assert booking_status_counts.sum() == len(bookings_raw)
assert flight_reference_counts.sum() == len(bookings_raw)
assert passenger_booking_reference_counts.sum() == len(bookings_raw)
assert bookings_standardized["status"].eq("UNKNOWN").eq(
    bookings_standardized[booking_status_flags].any(axis=1)
).all()

unresolved_flight_mask = ~bookings_standardized["flight_reference_status"].eq("accepted")
assert bookings_standardized.loc[unresolved_flight_mask, flight_attribute_columns].isna().all(axis=None)
assert bookings_standardized.loc[~unresolved_flight_mask, "departure_ts"].notna().all()
assert not accepted_flight_lookup.index.has_duplicates
assert flight_reference_lookup.loc[accepted_flight_lookup.index, "source_records"].eq(1).all()
assert bookings_standardized["booking_after_departure"].notna().eq(booking_date_evaluable).all()
assert (
    bookings_standardized["booking_after_departure"].eq(True).sum()
    + bookings_standardized["booking_after_departure"].eq(False).sum()
    + bookings_standardized["booking_after_departure"].isna().sum()
) == len(bookings_raw)
print("Booking preservation, reference, merge, and nullable-date checks passed.")
print("Python interpreter:", sys.executable)


Booking preservation, reference, merge, and nullable-date checks passed.
Python interpreter: /home/rrm/NeoStats/asg-airlines/.venv/bin/python


### Booking stage completed

The outputs are calculated from source data, not adjusted to match checkpoints. Bookings with unresolved references, uncertain passenger identities, unknown statuses, or date anomalies remain present for review. Flight and passenger reference status describe different relationships and must not be treated as a single overall validity flag.

The next stage standardizes payments and measures booking-level payment coverage. Existing flight and booking outputs remain unchanged. Passenger records are not cleaned, no final files are exported, and no dashboard is built.


## Payment standardization and booking-level payment coverage

A payment row represents a **payment record**, whereas the later `booking_payment_summary` has **one row per booking**. Multiple payments referencing one booking are not automatically duplicates. Keep every original payment record; flag exact duplicate groups, excess exact copies, repeated payment IDs, and normalization collisions without choosing winners. Totals describe the supplied records, so any duplicate/conflicting identities would require review before interpreting them as distinct transactions.

Create `payments_standardized` from `raw_tables["payments"].copy()`. Retain the source index and add `payment_source_row` (Excel row number = original zero-based index + 2 for the header). Preserve changed fields as `payment_id_raw`, `booking_id_raw`, `amount_raw`, and `payment_method_raw`. Reuse the existing nullable-string normalization and collision helpers. Missing and whitespace-only identifiers have separate flags, and both normalize to missing keys.

Outputs in this stage are aggregates. Booking lookups explicitly select only booking ID, source-row lineage, reporting status, and flight-reference status; no personal booking or passenger fields are joined or displayed.


In [24]:
payments_raw = raw_tables["payments"].copy()
payment_identifier_columns = ["payment_id", "booking_id"]
payments_standardized, payment_identifier_profile = cleaning.standardize_payments(payments_raw)


### Amounts: unknown is not zero

Parse the loaded amount using `Decimal(str(value).strip())`, preserving `amount_raw`. Constructing Decimal from the textual value avoids importing a float's binary expansion. Decimal values are numeric Python objects; pandas stores them in an object column so their decimal precision can be retained.

- `amount_missing`: source null as loaded; `amount_blank`: non-null empty/whitespace text.
- `amount_nonnumeric`: nonblank input that cannot be parsed as Decimal.
- `amount_nonfinite`: parsed NaN or infinity, which cannot be included in a monetary total.
- `amount_negative`: a finite signed value below zero; retain it in `amount` and flag for review. Do not assume it is a refund.
- `amount_zero`: a finite zero; zero remains usable.
- `amount_decimal_places` profiles fractional digits in the loaded representation; `amount_extra_precision` flags more than two for review. Two is a working precision checkpoint, not an assumed currency rule. Preserve extra precision exactly rather than rounding it away.

`amount` contains finite signed Decimal values; missing, blank, nonnumeric, and nonfinite values become null. `usable_amount` additionally excludes negative values pending clarification. `amount_usable` depends only on amount quality, not booking status, payment method, or reference matching. Extra precision is flagged but retained exactly. The precision profile describes loaded values, not Excel cell display formatting or recoverable trailing zeros.

Use a Decimal summation helper for both transaction and booking totals. It returns null when there are no usable values, including an all-null group. Arithmetic traps prevent silent precision rounding. Unknown amounts are never imputed as zero, mean, or median. A known zero is different from an unknown amount.

The known usable amount total has **currency unspecified**. It is not established net revenue: the extract does not establish settlement, refunds, fees, taxes, currency comparability, or revenue recognition. Booking cancellation alone neither removes a payment nor establishes a refund.


In [25]:
from decimal import Decimal
# Amount parsing and method flags were created together by standardize_payments.
# Keep these shared helpers available for the teaching edge checks below.
validate_payment_amounts = cleaning.validate_payment_amounts
sum_known_amounts = cleaning.sum_known_amounts
expected_payment_methods = cleaning.expected_payment_methods


### Booking references without changing payment validity

The earlier stage verified unique, non-missing standardized booking IDs. Check that precondition again; if it no longer holds, stop visibly rather than choosing a booking. Classify each payment reference as `missing`, `unmatched`, or `uniquely_matched`.

Left merge only the necessary, non-PII booking fields with `validate="many_to_one"`. This keeps all payment records and prevents lookup duplicates from multiplying rows. An actual matched booking with status UNKNOWN retains that reporting label; an unmatched payment has null booking attributes and a separate reference status. Booking status does not alter amount usability.


In [26]:
payments_standardized = cleaning.attach_booking_references(
    payments_standardized, bookings_standardized
)


### Aggregate transactions before joining to bookings

Group uniquely matched payments by booking ID first, calculating `payment_record_count`, `usable_amount_count`, and the exact `known_payment_amount`. Then left join those aggregates onto every booking with a one-to-one validation. Carry only booking ID, lineage, booking status, and flight-reference status into this new table. Never join unaggregated payment rows to a booking-level reporting table.

Fill absent **counts** with zero, and calculate `unusable_amount_count = payment_record_count - usable_amount_count`. Leave `known_payment_amount` null when no usable amount exists. Categorize coverage as:

- **no payment record in this extract**: zero matching records; not proof of nonpayment.
- **payment records but no usable amounts**: at least one record, zero usable amounts.
- **partial amount coverage**: both usable and unusable amounts.
- **complete amount coverage**: at least one record, all supplied amounts usable.

Complete coverage describes the amounts on the supplied records, not whether the fare was fully paid. A partial group's known subtotal is still incomplete. Missing/unmatched payment references remain in the payment table and a separate reference reconciliation so neither their records nor known amounts disappear from the overall total.


In [27]:
(
    booking_payment_summary, payment_reference_reconciliation, payments_by_booking,
) = cleaning.summarize_booking_payments(payments_standardized, bookings_standardized)
matched_payment_mask = payments_standardized["booking_reference_status"].eq("uniquely_matched")
matched_payments = payments_standardized.loc[
    matched_payment_mask, ["booking_id", "payment_source_row", "usable_amount"]
]
known_payment_total = sum_known_amounts(payments_standardized["usable_amount"])
known_matched_payment_total = sum_known_amounts(matched_payments["usable_amount"])
known_booking_payment_total = sum_known_amounts(booking_payment_summary["known_payment_amount"])


In [28]:
payment_row_summary = pd.Series({
    "Raw payment records": len(payments_raw),
    "Standardized payment records after merge": len(payments_standardized),
    "Records added by reference merge": len(payments_standardized) - len(payments_raw),
    "Distinct standardized payment IDs": payments_standardized["payment_id"].nunique(),
    "Distinct booking IDs referenced by payments": payments_standardized["booking_id"].nunique(),
    "Distinct matched booking IDs": matched_payments["booking_id"].nunique(),
    "Booking summary rows": len(booking_payment_summary),
    "Bookings with no payment record in this extract": int(booking_payment_summary["payment_record_count"].eq(0).sum()),
    "Bookings with multiple payment records": int(booking_payment_summary["payment_record_count"].gt(1).sum()),
}, name="count")
display(payment_row_summary.to_frame())
display(payment_identifier_profile)

payment_duplicate_flags = [
    "payment_exact_duplicate", "payment_exact_duplicate_excess",
    "payment_id_repeated_raw", "payment_id_repeated",
]
display(payments_standardized[payment_duplicate_flags].sum().astype(int).to_frame("flagged_records"))
payment_amount_flags = [
    "amount_missing", "amount_blank", "amount_nonnumeric", "amount_nonfinite",
    "amount_negative", "amount_zero", "amount_extra_precision", "amount_usable",
]
display(payments_standardized[payment_amount_flags].sum().astype(int).to_frame("payment_records"))
print("Unusable amounts:", int((~payments_standardized["amount_usable"]).sum()))
print("Usable-amount coverage (%):", payments_standardized["amount_usable"].mean() * 100)
payment_precision_counts = payments_standardized["amount_decimal_places"].value_counts().sort_index()
display(payment_precision_counts.rename_axis("supplied_decimal_places").to_frame("finite_amount_records"))
print("Known usable amount total (currency unspecified):", known_payment_total)
print("Known matched-payment amount:", known_matched_payment_total)
print("Known booking-level amount:", known_booking_payment_total)

payment_method_counts = payments_standardized["payment_method"].value_counts().reindex(
    [*expected_payment_methods, "UNKNOWN"], fill_value=0
)
display(payment_method_counts.rename_axis("payment_method").to_frame("payment_records"))
payment_method_flags = ["payment_method_missing", "payment_method_blank", "payment_method_unrecognized"]
display(payments_standardized[payment_method_flags].sum().astype(int).to_frame("flagged_records"))
display(payment_reference_reconciliation)

payment_coverage_categories = [
    "no payment record in this extract",
    "payment records but no usable amounts",
    "partial amount coverage",
    "complete amount coverage",
]
booking_payment_coverage_counts = booking_payment_summary["amount_coverage"].value_counts().reindex(
    payment_coverage_categories, fill_value=0
)
display(booking_payment_coverage_counts.rename_axis("amount_coverage").to_frame("bookings"))


,count
Raw payment records,1000
Standardized payment records after merge,1000
Records added by reference merge,0
Distinct standardized payment IDs,1000
Distinct booking IDs referenced by payments,637
Distinct matched booking IDs,637
Booking summary rows,1000
Bookings with no payment record in this extract,363
Bookings with multiple payment records,267


,raw_distinct_nonmissing,standardized_distinct_nonmissing,missing,blank,normalization_collision_rows,normalization_collision_keys
identifier,,,,,,
payment_id,1000,1000,0,0,0,0
booking_id,637,637,0,0,0,0


,flagged_records
payment_exact_duplicate,0
payment_exact_duplicate_excess,0
payment_id_repeated_raw,0
payment_id_repeated,0


,payment_records
amount_missing,48
amount_blank,0
amount_nonnumeric,30
amount_nonfinite,0
amount_negative,0
amount_zero,0
amount_extra_precision,0
amount_usable,922


Unusable amounts: 78
Usable-amount coverage (%): 92.2


...

Known usable amount total (currency unspecified): 7385142.98
Known matched-payment amount: 7385142.98
Known booking-level amount: 7385142.98


,payment_records
payment_method,
CARD,329
UPI,358
NETBANKING,313
UNKNOWN,0


,flagged_records
payment_method_missing,0
payment_method_blank,0
payment_method_unrecognized,0


,payment_record_count,usable_amount_count,known_payment_amount,unusable_amount_count
booking_reference_status,,,,
missing,0,0,NaN,0
unmatched,0,0,NaN,0
uniquely_matched,1000,922,7385142.98,78


,bookings
amount_coverage,
no payment record in this extract,363
payment records but no usable amounts,31
partial amount coverage,45
complete amount coverage,561


In [29]:
# Exercise the same money logic on cases absent from this extract.
amount_example = pd.Series(
    [None, " ", "INVALID", "Infinity", "NaN", "-2.50", "0", "0.10", "0.20", "1.234"],
    dtype=object,
)
amount_example_checks = validate_payment_amounts(amount_example)
assert amount_example_checks["amount_missing"].sum() == 1
assert amount_example_checks["amount_blank"].sum() == 1
assert amount_example_checks["amount_nonnumeric"].sum() == 1
assert amount_example_checks["amount_nonfinite"].sum() == 2
assert amount_example_checks["amount_negative"].sum() == 1
assert amount_example_checks["amount_zero"].sum() == 1
assert amount_example_checks["amount_extra_precision"].sum() == 1
assert amount_example_checks["amount_usable"].tolist() == [
    False, False, False, False, False, False, True, True, True, True,
]
assert amount_example_checks.loc[5, "amount"] == Decimal("-2.50")
assert amount_example_checks.loc[9, "usable_amount"] == Decimal("1.234")
assert pd.isna(sum_known_amounts(amount_example_checks.loc[:5, "usable_amount"]))
assert sum_known_amounts(amount_example_checks.loc[6:6, "usable_amount"]) == Decimal("0")
assert sum_known_amounts(amount_example_checks.loc[7:8, "usable_amount"]) == Decimal("0.30")
assert sum_known_amounts(amount_example_checks["usable_amount"]) == Decimal("1.534")

# Compare originals without printing source rows if a check fails.
payment_original_columns = payments_standardized[payments_raw.columns].copy()
for column in [*payment_identifier_columns, "amount", "payment_method"]:
    payment_original_columns[column] = payments_standardized[f"{column}_raw"]
assert payment_original_columns.equals(payments_raw), "Payment source values or lineage changed"
assert payments_standardized["payment_source_row"].tolist() == (payments_raw.index + 2).tolist()
assert payments_standardized["payment_source_row"].is_unique
assert len(payments_standardized) == len(payments_raw)
assert payments_standardized["usable_amount"].notna().equals(payments_standardized["amount_usable"])
assert payments_standardized.loc[
    ~matched_payment_mask, ["booking_status", "flight_reference_status", "booking_source_row"]
].isna().all(axis=None)

assert len(booking_payment_summary) == len(bookings_standardized)
assert booking_payment_summary["booking_id"].is_unique
assert booking_payment_summary["booking_id"].equals(bookings_standardized["booking_id"])
assert booking_payment_summary["booking_source_row"].equals(bookings_standardized["booking_source_row"])
assert booking_payment_summary["payment_record_count"].sum() == int(matched_payment_mask.sum())
assert booking_payment_summary["usable_amount_count"].sum() == matched_payments["usable_amount"].count()
assert booking_payment_summary["unusable_amount_count"].sum() == matched_payments["usable_amount"].isna().sum()
assert booking_payment_summary["known_payment_amount"].isna().eq(
    booking_payment_summary["usable_amount_count"].eq(0)
).all()
assert payment_reference_reconciliation["payment_record_count"].sum() == len(payments_raw)
assert payment_reference_reconciliation["usable_amount_count"].sum() == payments_standardized["amount_usable"].sum()
assert booking_payment_coverage_counts.sum() == len(bookings_standardized)
assert payment_method_counts.sum() == len(payments_raw)
assert payments_standardized["payment_method"].eq("UNKNOWN").eq(
    payments_standardized[payment_method_flags].any(axis=1)
).all()

# Null-aware equality: no known values is not a known zero total.
for transaction_total, grouped_total in [
    (known_matched_payment_total, known_booking_payment_total),
    (known_payment_total, sum_known_amounts(payment_reference_reconciliation["known_payment_amount"])),
]:
    assert (
        pd.isna(transaction_total) and pd.isna(grouped_total)
    ) or (
        pd.notna(transaction_total) and pd.notna(grouped_total)
        and transaction_total == grouped_total
    ), "Decimal monetary totals do not reconcile"

print("Payment preservation, decimal edge cases, join, count, and monetary reconciliation checks passed.")
print("Python interpreter:", sys.executable)


Payment preservation, decimal edge cases, join, count, and monetary reconciliation checks passed.
Python interpreter: /home/rrm/NeoStats/asg-airlines/.venv/bin/python


### Payment stage completed

All results above are calculated from this extract. Unknown amounts remain visible in usable/unusable counts and booking coverage, and records with unresolved booking references remain separately accounted for. No booking status is used to infer payment validity. Existing flight and booking tables remain unchanged.

The next stage assesses passenger quality and creates privacy-safe reporting views. Existing payment and booking-coverage objects remain unchanged. No final datasets are exported and no dashboard is built. Run the notebook from the beginning to recreate the tables.


## Passenger quality and privacy-safe reporting views

Work on `passengers_assessed`, a copy of the original passenger table. Keep original values, the source index, and `passenger_source_row` (loaded index + 2 for the Excel header). Preserve `passenger_id_raw` before reusing the existing nullable-string, whitespace-trimming, uppercase helper. A valid dimension key means a **non-missing normalized source ID**; no unsupported identifier-format rule is invented.

The reporting grain is one source identifier, **not one verified person**. Retain every source record in the restricted intermediate table. Count exact duplicate groups and excess copies before adding derived columns. Repeated IDs are counted separately; normalization collisions mean different original spellings collapse to one normalized key.

Compare every original field within each repeated normalized-ID group. Null versus a supplied value also counts as a disagreement (`nunique(dropna=False)`). Only aggregate counts of conflicting groups by field are displayed. No names, contact details, dates of birth, raw passenger IDs, or tokens are displayed.

Raw-field disagreements remain flagged even when normalized analytical values agree. Identity/contact conflicts include first name, last name, email, phone, Aadhaar, and date of birth; these are compared only in restricted memory and never carried into reporting views.


In [30]:
passengers_raw = raw_tables["passengers"].copy()
bookings_before_passenger_stage = bookings_standardized.copy(deep=True)
(
    passengers_assessed, passenger_dimension, passenger_field_conflicts,
) = privacy.assess_passengers(passengers_raw)
# Only aggregate field-conflict counts are displayed.
passenger_source_id_counts = passengers_assessed["passenger_id"].value_counts()
repeated_passenger_ids = passenger_source_id_counts.index[passenger_source_id_counts.gt(1)]
passenger_field_conflict_counts = passenger_field_conflicts.loc[repeated_passenger_ids].sum().astype(int)


### Reported age, gender, and conservative agreement

`MIN_REPORTED_AGE = 0` and `MAX_REPORTED_AGE = 120` are configurable plausibility limits, not proof of accuracy. Parse the supplied age numerically and flag missing, blank, nonnumeric, nonfinite, fractional, negative, and above-limit values. Only finite whole-number ages within the limits are usable. Age is **reported age**: do not derive it from date of birth or today's date. It has not been verified against an authoritative reference date.

The observed gender labels are M and F. The working map normalizes M/MALE to MALE and F/FEMALE to FEMALE after trimming and uppercasing. This is a dataset-specific label map, not a claim that these are the only possible genders. Missing, blank, explicit UNKNOWN, and other unrecognized labels have separate flags and report as UNKNOWN; extend the map only with validated source definitions. Do not infer gender from names or other personal details.

For each source ID, retain age only if **every row has a valid age and all numeric ages agree**. Retain gender only if every row resolves to the same recognized normalized label. No arbitrary row, majority vote, or inferred value resolves a disagreement. Null age and UNKNOWN gender represent unresolved analytical attributes. Raw identity/contact conflicts stay flagged regardless.

Age bands are non-overlapping intervals: 0–17, 18–29, 30–44, 45–59, and 60–120, with UNKNOWN for unresolved ages. Code uses left-inclusive, right-exclusive boundaries ending at maximum age + 1. If plausibility limits are changed, review the band definitions too.


In [31]:
# Shared plausibility/agreement rules; supplied age is not recalculated from birth dates.
assess_passenger_attributes = privacy.assess_passenger_attributes
agreed_attribute = privacy.agreed_attribute
MIN_REPORTED_AGE, MAX_REPORTED_AGE = privacy.MIN_REPORTED_AGE, privacy.MAX_REPORTED_AGE
passenger_age_flags = privacy.passenger_age_flags
passenger_gender_flags = privacy.passenger_gender_flags
age_band_edges, age_band_labels = privacy.age_band_edges, privacy.age_band_labels


### Persistent HMAC tokens and secret handling

Use standard-library HMAC-SHA-256 on the UTF-8 normalized passenger ID. The same normalized ID and key produce the same full-length token in both reporting tables. A missing ID produces no token. Tokens identify source keys; they do not resolve shared IDs, identity conflicts, or unmatched references.

The project-local `.secrets/passenger_hmac.key` holds 32 random bytes. The root `.gitignore` excludes `.secrets/` **before** any key is created. The loader requires this exclusion to remain the last active rule, reads an existing private key; explicit initialization creates the directory with mode 0700 and a new key with mode 0600. It rejects symlinks, inappropriate ownership/permissions, and invalid key length instead of replacing a questionable key. It never prints key material or puts it in notebook outputs. Do not display token columns or internal dimension indexes either.

Evaluator setup: retain the `.secrets/` exclusion as the last active rule in the project `.gitignore`, run `.venv/bin/python -m src.pipeline --init-key --key-file .secrets/passenger_hmac.key` explicitly from the checkout root, then run the notebook with that project's environment. Normal runs fail with setup instructions if the key is missing; initialization never overwrites an existing key. Their tokens will differ; reruns with the same key stay stable. Do not copy or commit someone else's key. Protect and retain your key for consistent linkage; replacing it changes the tokens.

This is **pseudonymization, not anonymization**. Tokens remain linkable, and reported ages, genders, routes, and booking/flight IDs can still support re-identification when combined with other information. The secret, raw workbook, raw tables, restricted intermediate tables, and saved working session require restricted access even after the reporting views are sanitized. Git exclusion prevents accidental tracking of this secret path; it does not sanitize other data.


In [32]:
import hmac
# Normal notebook runs only load the existing key; initialization is an explicit CLI action.
load_passenger_hmac_key = privacy.load_passenger_hmac_key
passenger_hmac_key = load_passenger_hmac_key(KEY_FILE)
# Bind this notebook's key without embedding it in code or outputs.
def passenger_token(normalized_id):
    return privacy.passenger_token(normalized_id, passenger_hmac_key)
print("Existing private HMAC key loaded; no key or token values displayed.")


Existing private HMAC key loaded; no key or token values displayed.


### Explicit reporting schemas and minimized joins

`passengers_reporting` selects only the token, agreed reported age/band, normalized gender, source record count, and explicit quality/conflict flags. Drop the internal source-ID index too. It excludes all raw IDs, names, dates of birth, identity documents, and contact details.

`bookings_reporting` starts from an explicit allowlist of booking/flight identifiers, timestamps, status, safe flight attributes, and quality/reference flags. Add a token derived from the same normalized booking passenger ID without changing `bookings_standardized`. Left join only agreed analytical attributes and passenger-conflict metadata from the token dimension using `validate="many_to_one"`. Unmatched analytical labels remain UNKNOWN and unresolved ages remain null; missing dimension conflict flags remain unknown rather than False.

Keep the existing `passenger_reference_status` and carry passenger source-record count and identity-conflict flags. The bookings referencing nonunique passenger IDs remain identifiable in aggregate: tokenization neither fixes nor hides those conflicts. No raw passenger IDs, passports, seats, emergency contacts, or raw PII columns enter the reporting schemas.


In [33]:
passengers_reporting, bookings_reporting = privacy.build_reporting_views(
    passenger_dimension, bookings_standardized, passenger_hmac_key
)
passenger_reporting_columns = privacy.passenger_reporting_columns
booking_reporting_columns = privacy.booking_reporting_columns


In [34]:
passenger_quality_summary = pd.Series({
    "Raw passenger records": len(passengers_raw),
    "Distinct raw passenger IDs": passengers_raw["passenger_id"].nunique(),
    "Valid normalized source IDs": passenger_source_id_counts.size,
    "Missing passenger IDs": int(passengers_assessed["passenger_id_missing"].sum()),
    "Blank passenger IDs": int(passengers_assessed["passenger_id_blank"].sum()),
    "Normalization collision records": int(passengers_assessed["passenger_id_normalization_collision"].sum()),
    "Normalization collision groups": int(passenger_dimension["passenger_id_normalization_collision"].sum()),
    "Exact duplicate group member rows": int(passengers_assessed["passenger_exact_duplicate"].sum()),
    "Excess exact duplicate copies": int(passengers_assessed["passenger_exact_duplicate_excess"].sum()),
    "Repeated normalized-ID groups": len(repeated_passenger_ids),
    "Rows in repeated-ID groups": int(passenger_source_id_counts.loc[repeated_passenger_ids].sum()),
    "Excess repeated-ID records": int((passenger_source_id_counts - 1).sum()),
    "Source IDs with record conflicts": int(passenger_dimension["record_conflict"].sum()),
    "Source IDs with identity/contact conflicts": int(passenger_dimension["identity_contact_conflict"].sum()),
    "Passenger reporting rows": len(passengers_reporting),
    "Booking reporting rows": len(bookings_reporting),
    "Bookings referencing nonunique passenger IDs": int(bookings_reporting["passenger_reference_status"].eq("existing-nonunique").sum()),
    "Bookings carrying passenger record conflicts": int(bookings_reporting["passenger_record_conflict"].sum()),
    "Bookings carrying passenger identity/contact conflicts": int(bookings_reporting["passenger_identity_contact_conflict"].sum()),
    "Unresolved dimension ages": int(passengers_reporting["age_unresolved"].sum()),
    "Unresolved dimension genders": int(passengers_reporting["gender_unresolved"].sum()),
}, name="count")
display(passenger_quality_summary.to_frame())
display(passenger_field_conflict_counts.rename_axis("source_field").to_frame("conflicting_repeated_ID_groups"))
display(passengers_assessed[
    passenger_age_flags + passenger_gender_flags
].sum().astype(int).to_frame("flagged_source_records"))
display(passengers_reporting["age_band"].value_counts().reindex(
    [*age_band_labels, "UNKNOWN"], fill_value=0
).rename_axis("age_band").to_frame("source_identifiers"))
display(passengers_reporting["gender"].value_counts().reindex(
    ["MALE", "FEMALE", "UNKNOWN"], fill_value=0
).rename_axis("normalized_gender").to_frame("source_identifiers"))


,count
Raw passenger records,1039
Distinct raw passenger IDs,1000
Valid normalized source IDs,1000
Missing passenger IDs,0
Blank passenger IDs,0
Normalization collision records,0
Normalization collision groups,0
Exact duplicate group member rows,0
Excess exact duplicate copies,0
Repeated normalized-ID groups,36


,conflicting_repeated_ID_groups
source_field,
passenger_id,0
first_name,11
last_name,27
age,0
gender,0
email,36
phone,36
aadhaar_id,36
date_of_birth,36


,flagged_source_records
age_missing,0
age_blank,0
age_nonnumeric,0
age_nonfinite,0
age_non_integer,0
age_negative,0
age_above_max,0
gender_missing,0
gender_blank,0
gender_explicit_unknown,0


,source_identifiers
age_band,
0–17,215
18–29,135
30–44,161
45–59,169
60–120,320
UNKNOWN,0


,source_identifiers
normalized_gender,
MALE,527
FEMALE,473
UNKNOWN,0


In [35]:
# Synthetic values exercise validity and agreement without displaying personal records.
age_examples = pd.Series([None, "", "bad", "inf", "NaN", 2.5, -1, MAX_REPORTED_AGE + 1, MIN_REPORTED_AGE, MAX_REPORTED_AGE], dtype=object)
gender_examples = pd.Series(["M", "male", "F", "female", None, "", "UNKNOWN", "unrecognized", "M", "F"])
attribute_examples = assess_passenger_attributes(age_examples, gender_examples)
assert attribute_examples["age_valid"].tolist() == [False] * 8 + [True, True]
assert attribute_examples["age_nonfinite"].sum() == 2
assert attribute_examples["age_non_integer"].sum() == 1
assert attribute_examples["age_negative"].sum() == 1
assert attribute_examples["age_above_max"].sum() == 1
assert attribute_examples["gender"].tolist() == [
    "MALE", "MALE", "FEMALE", "FEMALE", "UNKNOWN", "UNKNOWN", "UNKNOWN", "UNKNOWN", "MALE", "FEMALE",
]
assert agreed_attribute(pd.Series([20, 20])) == 20
assert pd.isna(agreed_attribute(pd.Series([20, 21])))
assert pd.isna(agreed_attribute(pd.Series([20, pd.NA])))
assert pd.isna(agreed_attribute(pd.Series(["MALE", "UNKNOWN"])))
assert agreed_attribute(attribute_examples.loc[:1, "gender"]) == "MALE"
assert pd.cut(
    pd.Series([MIN_REPORTED_AGE, 17, 18, 29, 30, 44, 45, 59, 60, MAX_REPORTED_AGE]),
    bins=age_band_edges, labels=age_band_labels, right=False,
).tolist() == [label for label in age_band_labels for _ in range(2)]

# Fail with generic messages rather than exposing source values in an assertion diff.
passenger_original_columns = passengers_assessed[passengers_raw.columns].copy()
passenger_original_columns["passenger_id"] = passengers_assessed["passenger_id_raw"]
passenger_original_columns["gender"] = passengers_assessed["gender_raw"]
assert passenger_original_columns.equals(passengers_raw), "Passenger originals or lineage changed"
assert passengers_assessed["passenger_source_row"].tolist() == (passengers_raw.index + 2).tolist()
assert bookings_standardized.equals(bookings_before_passenger_stage), "Existing booking data changed"
assert passenger_dimension.index.is_unique and passenger_dimension.index.notna().all()
assert len(passengers_reporting) == passengers_assessed["passenger_id"].nunique()
assert passengers_reporting["source_record_count"].sum() == passengers_assessed["passenger_id"].notna().sum()
assert passengers_reporting["passenger_token"].notna().all()
assert passengers_reporting["passenger_token"].is_unique
assert passengers_reporting["passenger_token"].str.fullmatch("[0-9a-f]{64}").all()
assert isinstance(passengers_reporting.index, pd.RangeIndex), "Raw source ID index leaked into reporting"

assert len(bookings_reporting) == len(bookings_standardized)
assert bookings_reporting.index.equals(bookings_standardized.index)
assert bookings_reporting["booking_id"].equals(bookings_standardized["booking_id"])
assert bookings_reporting["passenger_reference_status"].equals(bookings_standardized["passenger_reference_status"])
expected_booking_tokens = normalize_identifier(bookings_standardized["passenger_id"]).map(passenger_token)
assert bookings_reporting["passenger_token"].equals(expected_booking_tokens), "Booking token derivation differs"
dimension_booking_tokens = bookings_standardized["passenger_id"].map(passenger_dimension.index.to_series().map(passenger_token))
existing_passenger_reference = bookings_standardized["passenger_id"].isin(passenger_dimension.index)
assert bookings_reporting.loc[existing_passenger_reference, "passenger_token"].equals(
    dimension_booking_tokens.loc[existing_passenger_reference]
), "Passenger and booking tokens do not agree"
assert bookings_reporting["passenger_source_record_count"].gt(1).fillna(False).eq(
    bookings_standardized["passenger_reference_status"].eq("existing-nonunique")
).all()
expected_identity_conflict = bookings_standardized["passenger_id"].map(passenger_dimension["identity_contact_conflict"])
assert bookings_reporting["passenger_identity_contact_conflict"].fillna(False).eq(
    expected_identity_conflict.fillna(False)
).all()
assert hmac.compare_digest(passenger_hmac_key, load_passenger_hmac_key(KEY_FILE)), "Persistent key changed"
assert pd.isna(passenger_token(pd.NA))

for reporting_table, allowed_columns in [
    (passengers_reporting, passenger_reporting_columns),
    (bookings_reporting, booking_reporting_columns),
]:
    assert reporting_table.columns.tolist() == allowed_columns, "Reporting schema differs from allowlist"
    forbidden_columns = {
        "passenger_id", "first_name", "last_name", "name", "email", "phone",
        "aadhaar_id", "date_of_birth", "passport", "passport_number", "seat", "seat_number",
        "emergency_contact_name", "emergency_contact_phone",
    }
    assert not set(reporting_table.columns) & forbidden_columns, "Forbidden personal column in reporting"
    assert not any(column.endswith("_raw") for column in reporting_table.columns), "Raw column in reporting"

print("Passenger grain, preservation, agreement, HMAC reuse, join, and schema checks passed.")
print("Python interpreter:", sys.executable)
print("Passenger reporting schema:", passenger_reporting_columns)
print("Booking reporting schema:", booking_reporting_columns)


Passenger grain, preservation, agreement, HMAC reuse, join, and schema checks passed.
Python interpreter: /home/rrm/NeoStats/asg-airlines/.venv/bin/python
Passenger reporting schema: ['passenger_token', 'reported_age', 'age_band', 'gender', 'source_record_count', 'record_conflict', 'identity_contact_conflict', 'passenger_id_normalization_collision', 'age_conflict', 'gender_conflict', 'age_unresolved', 'gender_unresolved', 'age_missing', 'age_blank', 'age_nonnumeric', 'age_nonfinite', 'age_non_integer', 'age_negative', 'age_above_max', 'gender_missing', 'gender_blank', 'gender_explicit_unknown', 'gender_unrecognized']
Booking reporting schema: ['booking_id', 'flight_id', 'booking_ts', 'status', 'departure_ts', 'airline', 'source', 'destination', 'route', 'booking_exact_duplicate', 'booking_id_repeated', 'booking_id_missing', 'booking_id_blank', 'booking_id_normalization_collision', 'passenger_id_missing', 'passenger_id_blank', 'passenger_id_normalization_collision', 'flight_id_missing',

### Passenger reporting completed

Only aggregate quality/distribution results and schema metadata are displayed. All checkpoint counts are derived from data. The reporting views minimize personal data but remain pseudonymized and linkable; they are not permission to publish the dataset. Restricted raw/intermediate objects remain in memory and must not be displayed or exported casually.

The next stage builds and persists a restricted reporting model from these views. Nothing is published to GitHub and no dashboard is built.


## Reporting model and restricted analytical persistence

Build a full-refresh snapshot from the verified reporting objects using explicit column allowlists. The initial Power BI model uses FactFlights, FactBookings, and five dimensions. FactPayments is a separate transaction audit table; it is not needed in the initial Power BI relationship graph. Facts retain their own grains: accepted flight, booking, and payment record respectively.

Airline key 0 means UNKNOWN; other airline keys follow sorted canonical labels. Route key 0 means UNRESOLVED; real directional pairs receive keys in sorted source/destination order. Calendar keys are real YYYYMMDD dates. Passenger keys reuse the existing HMAC tokens. These keys are deterministic for this snapshot; changes to a full-refresh source can shift sorted route keys. A production incremental warehouse needs persistent dimension-key management.

Only accepted flight matches supply booking airline, route, or flight-date attribution. Unresolved bookings retain their source flight ID and reference classification, receive route 0 and airline 0, and have a null flight-date key. Accepted flights with unknown airline names also use airline 0; flight_reference_status distinguishes these cases. Date dimensions contain contiguous actual calendar dates between observed valid minimum and maximum dates, with no fake unknown date.

All eight model tables are persisted privately under data/processed/, which is Git-ignored. Tokens are permitted only in the restricted files and memory, not previews or logs. Exact reported age is omitted from DimPassenger, and no passenger attributes are repeated in FactBookings. See docs/reporting_model.md for the field dictionary and import instructions.


In [36]:
# One explicit contract is shared by notebook, CLI, CSV serialization, and SQLite DDL.
model_schemas = model.model_schemas
model_primary_keys = model.model_primary_keys
model_foreign_keys = model.model_foreign_keys
model_money_columns = model.model_money_columns


In [37]:
model_dimensions = model.build_dimensions(
    flights_standardized, bookings_reporting, passengers_reporting
)
DimAirline = model_dimensions["DimAirline"]
DimRoute = model_dimensions["DimRoute"]
DimFlightDate = model_dimensions["DimFlightDate"]
DimBookingDate = model_dimensions["DimBookingDate"]
DimPassenger = model_dimensions["DimPassenger"]


In [38]:
model_tables = model.build_model(
    flights_standardized, bookings_reporting, payments_standardized,
    booking_payment_summary, model_dimensions,
)
FactFlights = model_tables["FactFlights"]
FactBookings = model_tables["FactBookings"]
FactPayments = model_tables["FactPayments"]


### Relationships, dates, and monetary representation

Create only one-to-many, single-direction dimension-to-fact Power BI relationships:

- DimAirline.airline_key → FactFlights.airline_key and FactBookings.airline_key.
- DimRoute.route_key → FactFlights.route_key and FactBookings.route_key.
- DimFlightDate.date_key → FactFlights.flight_date_key and FactBookings.flight_date_key.
- DimBookingDate.date_key → FactBookings.booking_date_key.
- DimPassenger.passenger_token → FactBookings.passenger_token.

Do not directly relate FactFlights and FactBookings. FactPayments is stored for audit and later extensions, outside this initial graph. Its nullable matched_booking_id has a database foreign key to FactBookings while its source booking_id remains available to diagnose missing/unmatched references. No payment date is invented.

CSV monetary columns remain conventional decimal values for Power BI. Set known_payment_amount (and usable_amount if importing the audit table later) to **Fixed Decimal Number** in Power Query, explicitly using a locale that reads a dot decimal separator. This is a numeric type choice, not a currency declaration. Microsoft documents the type in [Power BI data types](https://learn.microsoft.com/en-us/power-bi/connect-data/desktop-data-types#fixed-decimal-number). Dates are ISO strings; set date columns to Date and timestamp columns to Date/Time. Keys/counts are Whole Number, tokens and natural IDs are Text, flags are 0/1 with blanks for unknown checks.

SQLite uses STRICT tables and integer monetary columns suffixed _minor at a scale of 100. The scale is a numeric representation, not an assumed currency. Additional fractional precision causes an explicit export failure instead of rounding. Unknown subtotals remain SQL NULL / empty CSV fields. Known payment amounts are incomplete observed amounts, not established net revenue.

Filtering real flight dates or resolved routes excludes bookings without that attribution; filtering booking dates is a separate operation. Keep reference-status coverage visible in reports so those excluded bookings do not disappear from the quality story. A contiguous date dimension fills calendar gaps only within real observed bounds; it inserts no fake unknown date.


In [39]:
# Shared checks reject schema drift and extra monetary precision before export.
MONEY_SCALE = persistence.MONEY_SCALE
minor_units = persistence.minor_units
model_csv_tables, model_sql_rows = persistence.prepare_model_tables(model_tables)
notebook_run["kpi_reconciliation"] = persistence.validate_model(model_tables)


### Staged, verified full refresh

The notebook and CLI use the same persistence functions. Build CSVs and a transactional SQLite database in a private staging directory; verify schemas, full contents, keys, nulls, dates, and exact money before replacing published files. Ordinary publication errors restore the previous files. The run summary is published last.

Separate file replacements are **not atomic as a set**. Do not run writers concurrently; readers must wait until the run completes. A crash during publication can leave a mixed set: rerun successfully and verify the manifest hashes before using it. No output is appended. The CLI and regression tests exercise reruns separately from this teaching notebook.


In [40]:
import sqlite3
from contextlib import closing

notebook_run["input_rows"] = {name: len(table) for name, table in raw_tables.items()}
notebook_run["quality_counts"] = quality_summary(
    raw_tables, flight_checks, bookings_standardized, payments_standardized,
    passengers_assessed, passenger_dimension,
)
assert persistence.sha256_file(SOURCE_FILE) == notebook_run["input"]["sha256"], "Source changed during execution"
model_database_file = persistence.publish_model(
    model_tables, PROJECT_ROOT / "data/processed", notebook_run
)
persisted_csv_tables, persisted_sql_tables = persistence.verify_persisted_model(
    model_database_file, model_tables
)
print("Validated candidate published; CSV and SQLite contents read back successfully.")


Validated candidate published; CSV and SQLite contents read back successfully.


In [41]:
persisted_flights = persisted_csv_tables["FactFlights"]
persisted_bookings = persisted_csv_tables["FactBookings"]
persisted_payments = persisted_csv_tables["FactPayments"]
assert persisted_flights["is_overnight"].sum() == flights_standardized["is_overnight"].sum()
assert abs(persisted_flights["duration_minutes"].mean() - flights_standardized["duration_minutes"].mean()) < 1e-10
assert persisted_bookings["status"].eq("CANCELLED").sum() == bookings_reporting["status"].eq("CANCELLED").sum()
assert persisted_bookings["status"].eq("UNKNOWN").sum() == bookings_reporting["status"].eq("UNKNOWN").sum()
assert persisted_bookings["known_payment_amount"].isna().eq(
    persisted_bookings["usable_amount_count"].eq(0)
).all()
unresolved_model_bookings = ~persisted_bookings["flight_reference_status"].eq("accepted")
assert persisted_bookings.loc[unresolved_model_bookings, "route_key"].eq(0).all()
assert persisted_bookings.loc[unresolved_model_bookings, "airline_key"].eq(0).all()
assert persisted_bookings.loc[unresolved_model_bookings, "flight_date_key"].isna().all()
assert persisted_bookings["flight_date_key"].notna().eq(~unresolved_model_bookings).all()
assert sum_known_amounts(persisted_payments["usable_amount"]) == known_payment_total
assert sum_known_amounts(persisted_bookings["known_payment_amount"]) == known_matched_payment_total

for table_name, observed in [
    ("DimFlightDate", flights_standardized["departure_ts"]),
    ("DimBookingDate", bookings_reporting["booking_ts"]),
]:
    dates = pd.to_datetime(persisted_csv_tables[table_name]["date"])
    expected_dates = pd.date_range(observed.dropna().min().normalize(), observed.dropna().max().normalize())
    assert dates.tolist() == expected_dates.tolist(), "Calendar coverage is not contiguous or has fake endpoints"
    assert persisted_csv_tables[table_name]["date_key"].tolist() == dates.dt.strftime("%Y%m%d").astype(int).tolist()

# Check enforcement and rollback with an invalid key; no row details are displayed.
with closing(sqlite3.connect(model_database_file)) as connection:
    connection.execute("PRAGMA foreign_keys = ON")
    try:
        connection.execute("BEGIN IMMEDIATE")
        connection.execute("UPDATE FactFlights SET airline_key = -1")
    except sqlite3.IntegrityError:
        connection.rollback()
    else:
        connection.rollback()
        raise AssertionError("SQLite did not reject an invalid foreign key")
    assert connection.execute("SELECT COUNT(*) FROM FactFlights WHERE airline_key = -1").fetchone()[0] == 0

model_row_counts = pd.Series({name: len(table) for name, table in persisted_csv_tables.items()}, name="persisted_rows")
display(model_row_counts.to_frame())
model_verification_summary = pd.Series({
    "Overnight accepted flights": int(persisted_flights["is_overnight"].sum()),
    "Mean accepted duration minutes": float(persisted_flights["duration_minutes"].mean()),
    "Cancelled bookings": int(persisted_bookings["status"].eq("CANCELLED").sum()),
    "Unknown-status bookings": int(persisted_bookings["status"].eq("UNKNOWN").sum()),
    "Usable payment amounts": int(persisted_payments["amount_usable"].sum()),
    "Known transaction amount, currency unspecified": str(sum_known_amounts(persisted_payments["usable_amount"])),
    "Known booking amount, currency unspecified": str(sum_known_amounts(persisted_bookings["known_payment_amount"])),
    "Unresolved booking flight attribution": int(unresolved_model_bookings.sum()),
    "Null booking payment subtotals": int(persisted_bookings["known_payment_amount"].isna().sum()),
}, name="verified_result")
display(model_verification_summary.to_frame())
for date_table in ("DimFlightDate", "DimBookingDate"):
    print(date_table, "range:", persisted_csv_tables[date_table]["date"].min(), "to", persisted_csv_tables[date_table]["date"].max())
print("Primary keys, foreign keys, exact money, nulls, date coverage, full refresh, and rollback checks passed.")


,persisted_rows
DimAirline,5
DimRoute,31
DimFlightDate,4
DimBookingDate,366
DimPassenger,1000
FactFlights,1002
FactBookings,1000
FactPayments,1000


,verified_result
Overnight accepted flights,122
Mean accepted duration minutes,164.53706
Cancelled bookings,314
Unknown-status bookings,75
Usable payment amounts,922
"Known transaction amount, currency unspecified",7385142.98
"Known booking amount, currency unspecified",7385142.98
Unresolved booking flight attribution,3
Null booking payment subtotals,394


DimFlightDate range: 2026-04-17 to 2026-04-20
DimBookingDate range: 2025-04-17 to 2026-04-17
Primary keys, foreign keys, exact money, nulls, date coverage, full refresh, and rollback checks passed.


### Model stage boundary

The eight CSVs and SQLite database are restricted, pseudonymized analytical outputs, not anonymous public datasets. No personal/token records are previewed. The source workbook, transformations, and HMAC secret remain unchanged. Known amounts remain incomplete observed amounts with currency unspecified.

Stop after model creation, persistence, documentation, and verification. Power BI construction, publishing, and GitHub pushes are outside this stage.
